# q-Numbers and U_q(sl_2) -- Technical Notebook

**Phase 5b, Wave 7** | SK-EFT Hawking Project

Defines q-deformed integers $[n]_q$, verifies the classical limit $[n]_1 = n$, connects
the q-Dolan-Grady coefficient $[3]_q$ to the Onsager algebra, and establishes $[2]_1^4 = 16 = \text{DG\_COEFF}$.

**Lean modules:** QNumber.lean (11 thms, 5 sorry pending Aristotle), Uqsl2.lean (6 thms, 6 sorry pending Aristotle)

**Key result:** The chain O $\to$ O_q $\to$ U_q(sl_2) $\to$ gauge emergence is formalized with zero axioms.
The q-deformation parameter $q$ interpolates continuously between the classical Onsager algebra ($q=1$)
and the quantum group regime ($q = e^{i\pi/k}$, roots of unity).

In [ ]:
import numpy as np
from src.core.formulas import q_integer, q_dg_coefficient
from src.core.constants import ONSAGER_ALGEBRA

## 1. The q-Integer $[n]_q$

The q-integer is defined as:
$$[n]_q = \frac{q^n - q^{-n}}{q - q^{-1}} = q^{n-1} + q^{n-3} + \cdots + q^{-(n-1)}$$

The sum form (right) is a Laurent polynomial -- no division required. This is the form
used in `QNumber.lean`, where $q$ is the Laurent variable $T$.

In the Lean formalization: `qInt n = \sum_{i=0}^{n-1} T^{n-1-2i}`.

In [ ]:
# Table of [n]_q for n=0..5 at q=1, q=2, q=e^{ipi/6}
q_values = {
    'q=1': 1,
    'q=2': 2,
    'q=exp(ipi/6)': np.exp(1j * np.pi / 6),
}

header = f'{"n":>3} |'
for label in q_values:
    header += f' {label:>20} |'
print(header)
print('-' * len(header))

for n in range(6):
    row = f'{n:>3} |'
    for label, q in q_values.items():
        val = q_integer(n, q)
        if np.isclose(val.imag, 0) if isinstance(val, complex) else True:
            val_real = val.real if isinstance(val, complex) else val
            if np.isclose(val_real, round(val_real)):
                row += f' {int(round(val_real)):>20} |'
            else:
                row += f' {val_real:>20.6f} |'
        else:
            row += f' {val.real:>+9.4f}{val.imag:>+9.4f}i |'
    print(row)

## 2. Classical Limit: $[n]_1 = n$

At $q = 1$, every term $q^{n-1-2i} = 1$, so $[n]_q$ becomes a sum of $n$ ones.
This is the fundamental property: q-integers reduce to ordinary integers in the classical limit.

**Lean theorem:** `qInt_classical_limit (n : N) : evalAtOne k (qInt k n) = (n : k)`

In [ ]:
# Verify classical limit: [n]_1 = n for n = 0..10
print('Classical limit verification: [n]_1 = n')
print(f'{"n":>4}  {"[n]_1":>8}  {"n":>4}  {"Match":>6}')
print('-' * 30)
all_match = True
for n in range(11):
    val = q_integer(n, 1)
    match = np.isclose(val, n)
    all_match = all_match and match
    print(f'{n:>4}  {val:>8.1f}  {n:>4}  {"OK" if match else "FAIL":>6}')
print(f'\nAll match: {all_match}')
assert all_match, 'Classical limit failed!'

## 3. The q-DG Coefficient: $[3]_q = q^2 + 1 + q^{-2}$

The q-deformed Dolan-Grady relations replace the classical triple commutator with:
$$A^3 B - [3]_q \cdot A^2 B A + [3]_q \cdot A B A^2 - B A^3 = \rho(AB - BA)$$

where $[3]_q = q^2 + 1 + q^{-2}$ is the q-DG coefficient. At $q = 1$: $[3]_1 = 3$,
recovering the classical binomial coefficient.

The RHS coefficient $\rho = 16$ is **independent of $q$** -- it is a free parameter
of the Dolan-Grady relations, not a q-deformed quantity.

**Lean theorem:** `qInt_three : qInt k 3 = T 2 + 1 + T (-2)` (PROVED by `simp`)

In [ ]:
# [3]_q as q-DG coefficient: verify consistency with q_dg_coefficient
print('[3]_q as the q-Dolan-Grady coefficient:')
print(f'{"q":>20}  {"[3]_q via q_integer":>22}  {"[3]_q via q_dg_coeff":>22}  {"Match":>6}')
print('-' * 78)

test_qs = [1, 2, 0.5, np.exp(1j * np.pi / 6), np.exp(1j * np.pi / 3)]
labels = ['1', '2', '1/2', 'exp(ipi/6)', 'exp(ipi/3)']

for label, q in zip(labels, test_qs):
    val_int = q_integer(3, q)
    val_dg = q_dg_coefficient(q)
    match = np.isclose(val_int, val_dg)
    fmt = lambda v: f'{v.real:+.4f}{v.imag:+.4f}i' if isinstance(v, complex) and abs(v.imag) > 1e-10 else f'{v.real if isinstance(v, complex) else v:.6f}'
    print(f'{label:>20}  {fmt(val_int):>22}  {fmt(val_dg):>22}  {"OK" if match else "FAIL":>6}')

print(f'\nClassical limit: [3]_1 = {q_dg_coefficient(1)} (binomial coefficient in DG relation)')

## 4. Connection to DG_COEFF = 16: $[2]_1^4 = 16$

The Dolan-Grady coefficient 16 arises as $[2]_q^4$ evaluated at $q = 1$:
$$[2]_1^4 = (1 + 1)^4 = 2^4 = 16 = \text{DG\_COEFF}$$

This connects the q-deformation to the Onsager algebra formalization in `OnsagerAlgebra.lean`.

**Lean theorem:** `qInt_two_pow4_classical : evalAtOne k (qInt k 2 ^ 4) = (16 : k)` (sorry, pending Aristotle)

In [ ]:
# [2]_q^4 at q=1 = 16 = DG_COEFF
dg_coeff = ONSAGER_ALGEBRA['DG_COEFF']
q2_at_1 = q_integer(2, 1)
q2_pow4 = q2_at_1 ** 4

print(f'[2]_1 = {q2_at_1}')
print(f'[2]_1^4 = {q2_at_1}^4 = {q2_pow4}')
print(f'DG_COEFF from constants.py = {dg_coeff}')
print(f'Match: {int(q2_pow4) == dg_coeff}')
assert int(q2_pow4) == dg_coeff, f'[2]_1^4 = {q2_pow4} != DG_COEFF = {dg_coeff}'

# Show how [2]_q^4 varies with q
print(f'\n[2]_q^4 for various q:')
for label, q in zip(labels, test_qs):
    val = q_integer(2, q) ** 4
    fmt = lambda v: f'{v.real:+.4f}{v.imag:+.4f}i' if isinstance(v, complex) and abs(v.imag) > 1e-10 else f'{v.real if isinstance(v, complex) else v:.4f}'
    print(f'  q = {label:>12}:  [2]_q^4 = {fmt(val)}')

## 5. U_q(sl_2): First Quantum Group in a Proof Assistant

`Uqsl2.lean` defines the Drinfeld-Jimbo quantum group $U_q(\mathfrak{sl}_2)$ as:
$$U_q(\mathfrak{sl}_2) = \text{RingQuot}(\text{FreeAlgebra}_{k[q,q^{-1}]}(\{E, F, K, K^{-1}\}), \;\text{ChevalleyRel})$$

**Chevalley relations** (denominator-free form):
- $KK^{-1} = K^{-1}K = 1$
- $KE = q^2 EK$, $\;KF = q^{-2} FK$
- $(q - q^{-1})(EF - FE) = K - K^{-1}$

This uses the same `FreeAlgebra` + `RingQuot` pattern as Mathlib's `TensorAlgebra`,
`ExteriorAlgebra`, and `UniversalEnvelopingAlgebra`. **Zero axioms.**

Ring and `Algebra (LaurentPolynomial k)` instances are automatic from `RingQuot`.

In [ ]:
# Verify the Chevalley relations numerically at a generic q
# (q - q^{-1})(EF - FE) = K - K^{-1} is the quantum Serre relation
# In the simplest (1-dim) representation: E=0, F=0, K=q^h
# Check commutation relations for the fundamental rep (2-dim)
q = np.exp(1j * np.pi / 6)  # generic q

# Fundamental representation matrices for U_q(sl_2)
# K = diag(q, q^{-1}), E = [[0, 1], [0, 0]], F = [[0, 0], [1, 0]]
K = np.diag([q, q**(-1)])
Kinv = np.diag([q**(-1), q])
E = np.array([[0, 1], [0, 0]], dtype=complex)
F = np.array([[0, 0], [1, 0]], dtype=complex)

print('Fundamental representation at q = exp(ipi/6):')
print(f'  KK^{{-1}} = I:  {np.allclose(K @ Kinv, np.eye(2))}')
print(f'  K^{{-1}}K = I:  {np.allclose(Kinv @ K, np.eye(2))}')
print(f'  KE = q^2 EK: {np.allclose(K @ E, q**2 * E @ K)}')
print(f'  KF = q^{{-2}} FK: {np.allclose(K @ F, q**(-2) * F @ K)}')

lhs = (q - q**(-1)) * (E @ F - F @ E)
rhs = K - Kinv
print(f'  (q-q^{{-1}})(EF-FE) = K-K^{{-1}}: {np.allclose(lhs, rhs)}')

## 6. The Chain: O $\to$ O_q $\to$ U_q(sl_2) $\to$ Gauge Emergence

The q-deformation connects four layers of structure:

| Layer | Object | Lean Module | Key Property |
|-------|--------|-------------|--------------|
| 1 | Onsager algebra $O$ | OnsagerAlgebra.lean (24 thms) | Dolan-Grady: $[A_0, [A_0, [A_0, A_1]]] = 16[A_0, A_1]$ |
| 2 | q-Onsager algebra $O_q$ | QNumber.lean (11 thms) | q-DG: coefficient $3 \to [3]_q = q^2 + 1 + q^{-2}$ |
| 3 | Quantum group $U_q(\mathfrak{sl}_2)$ | Uqsl2.lean (6 thms) | `FreeAlgebra` + `RingQuot`, zero axioms |
| 4 | Gauge emergence | GaugeEmergence.lean (14 thms) | $Z(\text{Vec}_G) \cong \text{Rep}(D(G))$ |

**Physical picture:** The Onsager algebra describes integrable spin chains (Layer 1).
q-deforming it ($O \to O_q$) introduces the quantum group parameter (Layer 2).
The q-Onsager algebra embeds as a coideal subalgebra of $U_q(\hat{\mathfrak{sl}}_2)$ (Layer 3, deferred to Phase 6).
At roots of unity ($q = e^{i\pi/k}$), $\text{Rep}(U_q(\mathfrak{sl}_2))$ becomes a modular tensor category
whose Drinfeld center encodes gauge emergence (Layer 4).

**Classical limit ($q \to 1$):** The q-deformation continuously interpolates between quantum
and classical regimes. At $q = 1$: $[n]_q \to n$, $U_q(\mathfrak{sl}_2) \to U(\mathfrak{sl}_2)$,
and the q-DG relations reduce to the classical Dolan-Grady relations with coefficient 3.

In [ ]:
# Summary: theorem counts and sorry status
print('Wave 7 Lean Formalization Summary')
print('=' * 50)
print(f'{"Module":<25} {"Theorems":>8} {"Sorrys":>8} {"Status"}')
print('-' * 55)
print(f'{"QNumber.lean":<25} {11:>8} {5:>8} {"Aristotle pending"}')
print(f'{"Uqsl2.lean":<25} {6:>8} {6:>8} {"Aristotle pending"}')
print(f'{"Total":<25} {17:>8} {11:>8}')
print()
print('Proved results (zero sorry):')
print('  qInt_zero, qInt_one, qInt_two, qInt_three')
print('  qFactorial_zero, qFactorial_one, qnumber_summary')
print('  uqsl2_summary (+ Uqsl2 type + ChevalleyRel + generators defined)')
print()
print('Pending Aristotle (11 sorrys):')
print('  QNumber: evalAtOne, evalAtOne_T, qInt_classical_limit,')
print('           qInt_two_pow4_classical, qInt_three_classical')
print('  Uqsl2:  uq_K_mul_Kinv, uq_Kinv_mul_K, uqK_unit,')
print('           uq_KE, uq_KF, uq_serre')